# V-Energy Grid 360 — Bronze (DE-01)

Este notebook lê os oito conjuntos do PRD sem corrigir valores nem aplicar
regras de negócio. O scanner definido neste notebook materializa cada registro lógico CSV/JSON em um arquivo
temporário do Volume, com texto bruto, ordinal e intervalo de linhas físicas.
Falhas de largura/parse ficam em `bronze.rejeitados`; a conta por arquivo/aba
compara a origem com aceitos + rejeitados.

Cada execução tem um ID e um schema de preparo próprios. Os dados validados
entram em tabelas Delta históricas de `bronze_hist`; as vistas `bronze.*`
filtram o lote ativo em `bronze.controle_publicacao`. A troca desse ponteiro é
uma única operação Delta, evitando mistura de versões entre tabelas.

O primeiro uso migra as tabelas Bronze legadas para o histórico. Execute essa
migração em janela controlada e confira o resultado antes de liberar leitores.
Em falha anterior à troca, o ponteiro continua no lote já publicado. As
origens ficam no Volume autorizado; nenhum dado bruto deve entrar no Git.

Pendências de aceite: executar no Databricks com a extração autorizada,
conferir contagens independentes, falha induzida e reexecução; registrar IDs
dos runs e evidências no PRD. Silver é responsável por limpeza e tratamento.

## 0. Dependências
`openpyxl` é necessário para percorrer as abas do Excel de inspeções e preservar a posição de cada linha.
Ele **não** vem garantido no ambiente serverless: em 19/10/2026 a execução parou na seção 6 com
`ImportError: Missing optional dependency 'openpyxl'`.

A célula seguinte é um comando mágico **de código** (não é markdown): rode-a sozinha antes de tudo.
No serverless ela instala no ambiente do notebook e o Python reinicia; depois disso, rode o notebook
do início (`Run all`) — o install vale para a sessão, então ao reabrir o notebook ele precisa rodar de novo.
Em cluster clássico, o `%pip` também reinicia sozinho; se o import falhar mesmo assim, execute
`dbutils.library.restartPython()` numa célula e rode o notebook do início.

In [ ]:
%pip install openpyxl

## 0.1. Parâmetros
Ajuste apenas `CATALOG` e `RAW` se necessário.

Layout esperado do Volume (suba os arquivos exatamente assim):
```
/Volumes/workspace/default/venergy/
├── unidades_consumidoras.csv
├── faturas.csv
├── medidores.csv
├── ocorrencias_rede.csv
├── geracao_distribuida.csv
├── inspecoes.xlsx
├── atendimentos.json
└── leituras/
    └── leituras_2022.csv ... leituras_2026.csv
```

> O autoteste sintético usa uma pasta temporária local ao processo e a remove ao terminar.
> A ingestão grava seus JSONL em `_bronze_preparo/{lote_id}` no Volume e os remove após a publicação bem-sucedida.

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4
import re

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("raw", "/Volumes/workspace/default/venergy")
dbutils.widgets.text("lote_origem", "")
CATALOG = dbutils.widgets.get("catalog").strip()
RAW = dbutils.widgets.get("raw").strip().rstrip("/")
LOTE_ORIGEM_PARAM = dbutils.widgets.get("lote_origem").strip()
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG):
    raise ValueError("catalog deve ser um identificador simples do Unity Catalog")
if not RAW.startswith("/Volumes/"):
    raise ValueError("raw deve apontar para um Volume autorizado")
SCHEMA = "bronze"
ALVO = f"{CATALOG}.{SCHEMA}"
DATA_CARGA = datetime.now(timezone.utc)
LOTE_ID = f"{DATA_CARGA.strftime('%Y%m%dT%H%M%SZ')}_{uuid4().hex[:8]}"
SCHEMA_STG = f"bronze_stg_{LOTE_ID.lower()}"
ALVO_STG = f"{CATALOG}.{SCHEMA_STG}"

In [ ]:
# Cria os schemas (idempotente): `ALVO` é a camada publicada; `ALVO_STG` é o preparo do lote.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ALVO}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ALVO_STG}")
print(f"Schemas prontos: {ALVO} e {ALVO_STG}")

## 1. Leitura e auditoria

O scanner abaixo separa registros lógicos antes do Spark e mantém o texto
original. As funções abaixo escrevem dados aceitos no preparo, rejeições com
motivo e o livro por origem. `_ordem_na_origem` é o ordinal do scanner, e
`_linha_fisica_inicial`/`_linha_fisica_final` localizam o trecho bruto.

In [ ]:
import hashlib

import csv
import json
from dataclasses import dataclass
from pathlib import Path


AUDIT_FIELDS = {
    "_arquivo_origem", "_ordem_na_origem", "_linha_fisica_inicial",
    "_linha_fisica_final", "_registro_bruto", "_falha_scanner",
    "_motivo_rejeicao", "_lote_id", "_lote_origem", "_versao_origem",
    "_id_registro", "_data_carga",
}


@dataclass(frozen=True)
class ScanSummary:
    stage_path: str
    source_records: int
    physical_lines: int
    malformed: int
    columns: tuple[str, ...] = ()


class _TrackedLines:
    def __init__(self, source):
        self.source = source
        self.number = 0
        self.current = []

    def __iter__(self):
        return self

    def __next__(self):
        line = self.source.readline()
        if not line:
            raise StopIteration
        self.number += 1
        self.current.append(line)
        return line


def _emit(target, row):
    target.write(json.dumps(row, ensure_ascii=False, separators=(",", ":")))
    target.write("\n")


def scan_csv(source_path, stage_path, origin, delimiter=","):
    """Conta registros lógicos e guarda texto/posição antes do parser Spark.

    Um registro com largura diferente é isolado; uma falha irrecuperável do
    próprio CSV conserva o restante consumido como um rejeitado. Se o encoding
    não puder ser interpretado, a carga falha em vez de trocar caracteres.
    """
    Path(stage_path).parent.mkdir(parents=True, exist_ok=True)
    with open(source_path, "r", encoding="utf-8-sig", newline="") as source, \
            open(stage_path, "w", encoding="utf-8", newline="") as target:
        tracked = _TrackedLines(source)
        # Aspas CSV são escapadas por duplicação; barra invertida pode ser dado literal.
        reader = csv.reader(tracked, delimiter=delimiter, quotechar='"', strict=True)
        try:
            header = next(reader)
        except (StopIteration, csv.Error) as exc:
            raise ValueError(f"Cabeçalho CSV ilegível: {origin}") from exc
        if not header or any(not col for col in header) or len(header) != len(set(header)):
            raise ValueError(f"Cabeçalho CSV vazio ou ambíguo: {origin}")
        if set(header) & AUDIT_FIELDS:
            raise ValueError(f"Cabeçalho de {origin} usa nomes reservados da auditoria")

        records = malformed = 0
        while True:
            tracked.current = []
            first_line = tracked.number + 1
            try:
                fields = next(reader)
            except StopIteration:
                break
            except csv.Error as exc:
                # O parser não tem ponto seguro para retomar após aspas abertas.
                # Todo o trecho consumido fica preservado; o restante não é publicado.
                records += 1
                malformed += 1
                _emit(target, {
                    "_arquivo_origem": origin,
                    "_ordem_na_origem": records,
                    "_linha_fisica_inicial": first_line,
                    "_linha_fisica_final": tracked.number,
                    "_registro_bruto": "".join(tracked.current),
                    "_falha_scanner": "".join(tracked.current),
                    "_motivo_rejeicao": f"CSV_SINTAXE: {type(exc).__name__}",
                })
                break
            records += 1
            raw = "".join(tracked.current)
            row = {
                "_arquivo_origem": origin,
                "_ordem_na_origem": records,
                "_linha_fisica_inicial": first_line,
                "_linha_fisica_final": tracked.number,
                "_registro_bruto": raw,
            }
            if len(fields) == len(header):
                row.update(zip(header, fields))
            else:
                malformed += 1
                row["_falha_scanner"] = raw
                row["_motivo_rejeicao"] = (
                    f"CSV_LARGURA: esperado={len(header)} recebido={len(fields)}"
                )
            _emit(target, row)
        # Uma falha de sintaxe que não consumiu o EOF não pode ser contabilizada
        # com segurança. Não exponha uma reconciliação enganosa como completa.
        if source.read(1):
            raise ValueError(f"CSV {origin}: falha de sintaxe antes do fim do arquivo")
        return ScanSummary(stage_path, records, tracked.number, malformed,
                           tuple(header))


def _unique_pairs(pairs):
    obj = {}
    for key, value in pairs:
        if key in obj:
            raise ValueError(f"chave JSON repetida: {key}")
        obj[key] = value
    return obj


def scan_json_array(source_path, stage_path, origin):
    """Separa objetos de um array JSON sem usar a contagem do parser Spark.

    O scanner acompanha strings e aninhamento, logo vírgulas e colchetes dentro
    dos objetos não criam registros artificiais. Preserva cada objeto bruto.
    """
    Path(stage_path).parent.mkdir(parents=True, exist_ok=True)
    state = "open"
    depth = line = 1
    start_line = 1
    quoted = escaped = False
    last_char = None
    current = []
    records = malformed = 0
    with open(source_path, "r", encoding="utf-8-sig", newline="") as source, \
            open(stage_path, "w", encoding="utf-8", newline="") as target:
        while chunk := source.read(1024 * 1024):
            for char in chunk:
                last_char = char
                at_line = line
                if char == "\n":
                    line += 1
                if state == "open":
                    if char.isspace():
                        continue
                    if char != "[":
                        raise ValueError(f"JSON {origin}: raiz não é array")
                    state = "value_or_close"
                elif state in ("value_or_close", "value"):
                    if char.isspace():
                        continue
                    if char == "]":
                        if state == "value":
                            raise ValueError(f"JSON {origin}: vírgula final na linha {at_line}")
                        state = "done"
                    elif char == "{":
                        state = "object"
                        depth = 1
                        start_line = at_line
                        current = [char]
                    else:
                        raise ValueError(f"JSON {origin}: esperado objeto na linha {at_line}")
                elif state == "object":
                    current.append(char)
                    if quoted:
                        if escaped:
                            escaped = False
                        elif char == "\\":
                            escaped = True
                        elif char == '"':
                            quoted = False
                    elif char == '"':
                        quoted = True
                    elif char in "{[":
                        depth += 1
                    elif char in "}]":
                        depth -= 1
                        if depth == 0:
                            records += 1
                            raw = "".join(current)
                            row = {
                                "_arquivo_origem": origin,
                                "_ordem_na_origem": records,
                                "_linha_fisica_inicial": start_line,
                                "_linha_fisica_final": at_line,
                                "_registro_bruto": raw,
                            }
                            try:
                                obj = json.loads(raw, object_pairs_hook=_unique_pairs)
                                if not isinstance(obj, dict):
                                    raise ValueError("elemento não é objeto")
                                if set(obj) & AUDIT_FIELDS:
                                    raise ValueError("campo de origem colide com auditoria")
                                row.update(obj)
                            except (ValueError, json.JSONDecodeError) as exc:
                                malformed += 1
                                row["_falha_scanner"] = raw
                                row["_motivo_rejeicao"] = f"JSON_SINTAXE: {exc}"
                            _emit(target, row)
                            current = []
                            state = "separator"
                elif state == "separator":
                    if char.isspace():
                        continue
                    if char == ",":
                        state = "value"
                    elif char == "]":
                        state = "done"
                    else:
                        raise ValueError(f"JSON {origin}: separador inválido na linha {at_line}")
                elif state == "done" and not char.isspace():
                    raise ValueError(f"JSON {origin}: conteúdo após o array")
        if state != "done":
            raise ValueError(f"JSON {origin}: array incompleto na linha {line}")
    physical_lines = line - 1 if last_char == "\n" else line
    return ScanSummary(stage_path, records, physical_lines, malformed)
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, LongType, StringType, StructField, StructType, TimestampType

COLUNA_CORROMPIDA = "_falha_scanner"
reconciliacao = []
rejeitados_coletados = []
ORIGENS_LIDAS = {}
ARQUIVOS_IDENTIDADE = {}


def registrar_falha(etapa, origem, exc):
    """Deixa falhas diagnosticáveis sem incluir valores da origem na mensagem."""
    causa = f": {exc}" if isinstance(exc, ValueError) else ""
    detalhe = f"{etapa} ({origem}): {type(exc).__name__}{causa}"[:400]
    seguro = detalhe.replace("'", "''")
    try:
        spark.sql(f"""
            UPDATE {ALVO}.controle_lote SET status = 'FALHOU', observacao = '{seguro}'
            WHERE _lote_id = '{LOTE_ID}'
        """)
    except Exception as audit_exc:
        print(f"Falha ao registrar estado do lote: {type(audit_exc).__name__}")


def add_ordem_origem(df):
    """Somente Excel sem posição de origem recebe uma ordem técnica de leitura."""
    janela = Window.partitionBy("_arquivo_origem").orderBy(F.col("_id_registro"))
    return df.withColumn("_ordem_na_origem", F.row_number().over(janela))


def add_audit(df, origem):
    df = df.withColumn("_arquivo_origem", F.lit(origem))
    if "_id_registro" not in df.columns:
        df = df.withColumn("_id_registro", F.monotonically_increasing_id())
    if "_ordem_na_origem" not in df.columns:
        df = add_ordem_origem(df)
    return (df.withColumn("_lote_id", F.lit(LOTE_ID))
              .withColumn("_lote_origem", F.lit(LOTE_ORIGEM))
              .withColumn("_versao_origem", F.lit(VERSAO_ORIGEM))
              .withColumn("_data_carga", F.lit(DATA_CARGA).cast("timestamp")))


def write_bronze(df, tabela, origem):
    out = add_audit(df, origem)
    try:
        (out.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
            .saveAsTable(f"{ALVO_STG}.{tabela}"))
        count = spark.table(f"{ALVO_STG}.{tabela}").count()
    except Exception as exc:
        registrar_falha("gravação", origem, exc)
        raise
    print(f"OK  {ALVO_STG}.{tabela}: {count:,} registros no preparo")
    return count


def _stage_path(relative):
    slug = hashlib.sha256(relative.encode("utf-8")).hexdigest()[:16]
    return f"{RAW}/_bronze_preparo/{LOTE_ID}/{slug}.jsonl"


def read_csv(relative, sep=","):
    """Lê registros lógicos com posição física e rejeita largura sem perder campos."""
    try:
        audit = scan_csv(f"{RAW}/{relative}", _stage_path(relative), relative, sep)
    except Exception as exc:
        registrar_falha("leitura CSV", relative, exc)
        raise
    ORIGENS_LIDAS[relative] = audit
    fields = [StructField(c, StringType(), True) for c in audit.columns]
    fields.extend([
        StructField("_arquivo_origem", StringType(), False),
        StructField("_ordem_na_origem", LongType(), False),
        StructField("_linha_fisica_inicial", LongType(), False),
        StructField("_linha_fisica_final", LongType(), False),
        StructField("_registro_bruto", StringType(), True),
        StructField("_falha_scanner", StringType(), True),
        StructField("_motivo_rejeicao", StringType(), True),
    ])
    return spark.read.schema(StructType(fields)).json(audit.stage_path)


def read_json_com_corrupt(relative):
    """O scanner independente separa o array e guarda o objeto original."""
    try:
        audit = scan_json_array(f"{RAW}/{relative}", _stage_path(relative), relative)
    except Exception as exc:
        registrar_falha("leitura JSON", relative, exc)
        raise
    ORIGENS_LIDAS[relative] = audit
    df = spark.read.option("samplingRatio", 1.0).json(audit.stage_path)
    for name in ("_falha_scanner", "_motivo_rejeicao"):
        if name not in df.columns:
            df = df.withColumn(name, F.lit(None).cast("string"))
    return df


def separar_validos_rejeitados(df, tabela, arquivo_origem):
    validos = df.filter(F.col(COLUNA_CORROMPIDA).isNull()).drop(
        COLUNA_CORROMPIDA, "_motivo_rejeicao")
    rejeitados = (df.filter(F.col(COLUNA_CORROMPIDA).isNotNull())
                  .select(F.col("_registro_bruto").alias("_registro"),
                          F.col("_motivo_rejeicao").alias("_motivo"),
                          "_ordem_na_origem", "_linha_fisica_inicial",
                          "_linha_fisica_final")
                  .withColumn("_tabela_origem", F.lit(tabela))
                  .withColumn("_origem_rejeitados", F.lit(arquivo_origem))
                  .withColumn("_lote_id", F.lit(LOTE_ID))
                  .withColumn("_lote_origem", F.lit(LOTE_ORIGEM))
                  .withColumn("_versao_origem", F.lit(VERSAO_ORIGEM))
                  .withColumn("_data_carga", F.lit(DATA_CARGA).cast("timestamp")))
    return validos, rejeitados


def contar_linhas_arquivo(*relatives):
    """Contagem física diagnóstica; a base de reconciliação é o registro lógico."""
    return sum(ORIGENS_LIDAS[r].physical_lines for r in relatives)


def registrar_reconciliacao(tabela, arquivo_origem, linhas_arquivo, cabecalhos,
                            rejeitadas, gravadas, status=None, linhas_dados=None,
                            aba_origem=None, objetos_interpretados=None):
    audit = ORIGENS_LIDAS.get(arquivo_origem)
    if audit is not None:
        linhas_dados = audit.source_records
        linhas_arquivo = audit.physical_lines
    elif linhas_dados is None and linhas_arquivo is not None:
        linhas_dados = linhas_arquivo - cabecalhos
    lidos = gravadas + rejeitadas
    delta = None if linhas_dados is None else linhas_dados - lidos
    if status is None:
        status = ("INVESTIGAR" if delta is None or delta != 0 or rejeitadas < 0
                  else "OK_COM_REJEICOES" if rejeitadas else "OK")
    origem_df = spark.table(f"{ALVO_STG}.{tabela}")
    if aba_origem is not None:
        origem_df = origem_df.where(F.col("_aba_origem") == aba_origem)
    elif tabela == "leituras":
        origem_df = origem_df.where(F.col("_arquivo_origem") == arquivo_origem)
    campos = [c for c in origem_df.columns if not c.startswith("_")]
    assinatura = F.sha2(F.to_json(F.struct(*[F.col("`" + c.replace("`", "``") + "`")
                                              for c in campos])), 256) if campos else None
    duplicatas = (gravadas - origem_df.select(assinatura.alias("_assinatura")).distinct().count()
                  if assinatura is not None else 0)
    identidade = ARQUIVOS_IDENTIDADE.get(arquivo_origem, {})
    reconciliacao[:] = [r for r in reconciliacao if
                        (r["_tabela"], r["_arquivo_origem"], r["_aba_origem"])
                        != (tabela, arquivo_origem, aba_origem)]
    reconciliacao.append(dict(
        _lote_id=LOTE_ID, _lote_origem=LOTE_ORIGEM,
        _versao_origem=VERSAO_ORIGEM,
        _tabela=tabela, _arquivo_origem=arquivo_origem,
        _aba_origem=aba_origem, hash_sha256=identidade.get("hash_sha256"),
        linhas_arquivo=linhas_arquivo, cabecalhos=cabecalhos,
        linhas_dados=linhas_dados, objetos_interpretados=objetos_interpretados,
        lidos=lidos, rejeitadas=rejeitadas, gravadas=gravadas,
        duplicatas_detectadas=duplicatas, corrigidos=0, quarentenados=rejeitadas,
        descartados=0, delta=delta, status=status, _data_carga=DATA_CARGA))
    name = arquivo_origem if aba_origem is None else f"{arquivo_origem}[{aba_origem}]"
    print(f"REC {name}: origem={linhas_dados} lidos={lidos} aceitos={gravadas} "
          f"rejeitados={rejeitadas} duplicatas={duplicatas} delta={delta} {status}")

## 1.1. Validação por origem

O scanner registra todos os registros lógicos, inclusive campos multilinha, e
preserva os rejeitados com o texto bruto. A reconciliação usa essa contagem
independente do parser Spark. A verificação no Databricks deve comparar os
resultados da execução com os arquivos autorizados do Volume.

In [ ]:
# Teste sintético do scanner que será usado nas fontes deste lote.
import tempfile


def autoteste_scanners():
    with tempfile.TemporaryDirectory(prefix="bronze-autoteste-") as pasta:
        pasta = Path(pasta)
        csv_origem, csv_stage = pasta / "largura.csv", pasta / "largura.jsonl"
        csv_origem.write_text("id,nome\n1,ok\n3,ok,EXTRA\n7\n", encoding="utf-8")
        csv_resumo = scan_csv(str(csv_origem), str(csv_stage), "_autoteste/largura.csv")
        csv_registros = [json.loads(l) for l in csv_stage.read_text(encoding="utf-8").splitlines()]
        csv_rejeitados = [r for r in csv_registros if r.get("_falha_scanner")]
        csv_ok = (
            csv_resumo.source_records == 3
            and csv_resumo.malformed == 2
            and len(csv_registros) == 3
            and len(csv_rejeitados) == 2
            and csv_rejeitados[0]["_linha_fisica_inicial"] == 3
            and csv_rejeitados[1]["_linha_fisica_inicial"] == 4
            and "3,ok,EXTRA" in csv_rejeitados[0]["_registro_bruto"]
        )

        json_origem, json_stage = pasta / "objetos.json", pasta / "objetos.jsonl"
        validos_json = [{"id": 1, "texto": 'vírgula, colchete ] e aspas "ok"'}, {"id": 2}]
        json_texto = json.dumps(validos_json, ensure_ascii=False)[:-1] + ', {"id":,}]'
        json_origem.write_text(json_texto, encoding="utf-8")
        json_resumo = scan_json_array(str(json_origem), str(json_stage), "_autoteste/objetos.json")
        json_registros = [json.loads(l) for l in json_stage.read_text(encoding="utf-8").splitlines()]
        json_rejeitados = [r for r in json_registros if r.get("_falha_scanner")]
        json_ok = (
            json_resumo.source_records == 3
            and json_resumo.malformed == 1
            and len(json_registros) == 3
            and len(json_rejeitados) == 1
            and json_registros[0]["texto"] == validos_json[0]["texto"]
            and json_rejeitados[0]["_registro_bruto"] == '{"id":,}'
        )

        lidos = csv_resumo.source_records + json_resumo.source_records
        rejeitadas = csv_resumo.malformed + json_resumo.malformed
        return {
            "status": "OK" if csv_ok and json_ok else "FALHA",
            "detalhe": (f"CSV: lidos={csv_resumo.source_records}, rejeitados={csv_resumo.malformed}; "
                        f"JSON: objetos={json_resumo.source_records}, rejeitados={json_resumo.malformed}"),
            "lidos": lidos,
            "aceitos": lidos - rejeitadas,
            "rejeitados": rejeitadas,
            "objetos_json": json_resumo.source_records,
        }


In [ ]:
_RESULTADO_AUTOTESTE = autoteste_scanners()
print(f"AUTOTESTE scanners Bronze: {_RESULTADO_AUTOTESTE['status']} — {_RESULTADO_AUTOTESTE['detalhe']}")
if _RESULTADO_AUTOTESTE["status"] != "OK":
    raise RuntimeError("Autoteste sintético dos scanners falhou; lote não será publicado.")


## 1.2. Controle da execução e identidade da origem

`_lote_id` identifica a execução; `hash_sha256` identifica a versão de cada
arquivo. A carga é bloqueada se o hash não puder ser lido. O preparo desta
execução usa schema próprio para não sobrescrever outra tentativa.

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {ALVO}.controle_lote (
  _lote_id STRING NOT NULL,
  iniciado_em TIMESTAMP NOT NULL,
  publicado_em TIMESTAMP,
  status STRING NOT NULL,
  tabelas_publicadas INT,
  linhas_publicadas BIGINT,
  observacao STRING
) USING DELTA
""")
spark.sql(f"""
INSERT INTO {ALVO}.controle_lote (_lote_id, iniciado_em, status)
SELECT '{LOTE_ID}', current_timestamp(), 'EM_ANDAMENTO'
""")
print(f"Lote {LOTE_ID}: EM_ANDAMENTO")

ARQUIVOS_DO_LOTE = [
    "unidades_consumidoras.csv", "medidores.csv", "ocorrencias_rede.csv",
    "geracao_distribuida.csv", "faturas.csv", "atendimentos.json", "inspecoes.xlsx",
] + [f"leituras/leituras_{ano}.csv" for ano in range(2022, 2027)]


def info_arquivo(caminho_relativo):
    """Identidade do arquivo de origem: nome, tamanho, modificação e SHA-256 (streaming)."""
    caminho = f"{RAW}/{caminho_relativo}"
    info = dbutils.fs.ls(caminho)[0]
    hash_sha256, status_hash = None, "INDISPONIVEL"
    try:
        h = hashlib.sha256()
        with open(caminho, "rb") as f:
            for bloco in iter(lambda: f.read(1024 * 1024), b""):
                h.update(bloco)
        hash_sha256, status_hash = h.hexdigest(), "OK"
    except Exception as e:
        # Sem hash não há prova da versão carregada; o lote não é publicado.
        raise RuntimeError(f"Hash da origem {caminho_relativo} indisponível: {type(e).__name__}") from e
    return dict(
        _lote_id=LOTE_ID,
        caminho=caminho_relativo,
        nome=caminho_relativo.split("/")[-1],
        tamanho_bytes=info.size,
        modificado_em=datetime.fromtimestamp(info.modificationTime / 1000, tz=timezone.utc),
        hash_sha256=hash_sha256,
        status_hash=status_hash,
        _data_carga=DATA_CARGA)


try:
    arquivos_lote = [info_arquivo(c) for c in ARQUIVOS_DO_LOTE]
except Exception as exc:
    registrar_falha("identidade dos arquivos", "lote", exc)
    raise

for a in arquivos_lote:
    print(f"ARQ {a['caminho']:<40} {a['tamanho_bytes']:>12,} B  "
          f"sha256={(a['hash_sha256'] or '')[:16]:<16} {a['status_hash']}")
VERSAO_ORIGEM = hashlib.sha256("\n".join(
    f"{a['caminho']}:{a['hash_sha256']}" for a in sorted(arquivos_lote, key=lambda x: x['caminho'])
).encode("utf-8")).hexdigest()
LOTE_ORIGEM = LOTE_ORIGEM_PARAM or VERSAO_ORIGEM[:16]
for a in arquivos_lote:
    a["_lote_origem"] = LOTE_ORIGEM
    a["_versao_origem"] = VERSAO_ORIGEM
ARQUIVOS_IDENTIDADE = {a["caminho"]: a for a in arquivos_lote}
print(f"Versão da origem: {VERSAO_ORIGEM}; lote de origem: {LOTE_ORIGEM}")

reconciliacao.append(dict(
    _lote_id=LOTE_ID, _lote_origem=LOTE_ORIGEM, _versao_origem=VERSAO_ORIGEM,
    _tabela="_autoteste_scanners", _arquivo_origem="_autoteste_sintetico",
    _aba_origem=None, hash_sha256=None, linhas_arquivo=None, cabecalhos=0,
    linhas_dados=_RESULTADO_AUTOTESTE["lidos"],
    objetos_interpretados=_RESULTADO_AUTOTESTE["objetos_json"],
    lidos=_RESULTADO_AUTOTESTE["lidos"],
    rejeitadas=_RESULTADO_AUTOTESTE["rejeitados"],
    gravadas=_RESULTADO_AUTOTESTE["aceitos"], duplicatas_detectadas=0,
    corrigidos=0, quarentenados=_RESULTADO_AUTOTESTE["rejeitados"],
    descartados=0, delta=0, status="AUTOTESTE_OK", _data_carga=DATA_CARGA))

## 2. CSVs simples (separador vírgula)
`total_lida = bruto.count()` é seguro (nenhum filtro na coluna de corrupção ainda).
`rejeitadas` sai por aritmética (`total_lida - gravadas`), nunca de `rejeitados.count()`.

In [ ]:
for arquivo, tabela in [
    ("unidades_consumidoras.csv", "unidades_consumidoras"),
    ("medidores.csv",             "medidores"),
    ("ocorrencias_rede.csv",      "ocorrencias_rede"),
    ("geracao_distribuida.csv",   "geracao_distribuida"),
]:
    bruto = read_csv(arquivo)
    total_lida = bruto.count()
    validos, rejeitados = separar_validos_rejeitados(bruto, tabela, arquivo)
    gravadas = write_bronze(validos, tabela, arquivo)
    rejeitadas = total_lida - gravadas
    rejeitados_coletados.append(rejeitados)
    registrar_reconciliacao(tabela, arquivo,
                            linhas_arquivo=contar_linhas_arquivo(arquivo),
                            cabecalhos=1,
                            rejeitadas=rejeitadas,
                            gravadas=gravadas)

## 3. `faturas.csv` — separador **ponto e vírgula**

In [ ]:
bruto = read_csv("faturas.csv", sep=";")
total_lida = bruto.count()
validos, rejeitados = separar_validos_rejeitados(bruto, "faturas", "faturas.csv")
gravadas = write_bronze(validos, "faturas", "faturas.csv")
rejeitadas = total_lida - gravadas
rejeitados_coletados.append(rejeitados)
registrar_reconciliacao("faturas", "faturas.csv",
                        linhas_arquivo=contar_linhas_arquivo("faturas.csv"),
                        cabecalhos=1,
                        rejeitadas=rejeitadas,
                        gravadas=gravadas)

## 4. Cinco arquivos anuais de leituras

Cada arquivo é varrido e reconciliado separadamente. A união mantém a origem,
o ordinal e a posição física de cada registro.

In [ ]:
arquivos_leituras = [f"leituras/leituras_{ano}.csv" for ano in range(2022, 2027)]

leituras_validas = None
leituras_rejeitadas = None
lidas_por_arquivo = {}
for arquivo in arquivos_leituras:
    bruto = read_csv(arquivo)
    lidas_por_arquivo[arquivo] = bruto.count()
    validos, rejeitados = separar_validos_rejeitados(bruto, "leituras", arquivo)
    validos = validos.withColumn("_arquivo_origem", F.lit(arquivo))
    leituras_validas = (validos if leituras_validas is None
                        else leituras_validas.unionByName(validos, allowMissingColumns=True))
    leituras_rejeitadas = (rejeitados if leituras_rejeitadas is None
                           else leituras_rejeitadas.unionByName(rejeitados))

# O scanner já registrou o ordinal e a linha física de cada arquivo anual.
leituras_validas = (leituras_validas
                     .withColumn("_lote_id", F.lit(LOTE_ID))
                     .withColumn("_lote_origem", F.lit(LOTE_ORIGEM))
                     .withColumn("_versao_origem", F.lit(VERSAO_ORIGEM))
                     .withColumn("_id_registro", F.monotonically_increasing_id())
                     .withColumn("_data_carga", F.lit(DATA_CARGA).cast("timestamp")))

(leituras_validas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{ALVO_STG}.leituras"))
gravadas = spark.table(f"{ALVO_STG}.leituras").count()   # conta a tabela (não reparseia os CSVs)
print(f"OK  {ALVO_STG}.leituras  {gravadas:>10,} linhas (preparo)")

# Reconciliação POR ARQUIVO: cada ano tem sua própria linha (o §4.3 pede "para cada fonte e lote").
gravadas_por_arquivo = {r["_arquivo_origem"]: r["count"] for r in
                        spark.table(f"{ALVO_STG}.leituras")
                             .groupBy("_arquivo_origem").count().collect()}
rejeitados_coletados.append(leituras_rejeitadas)
for arquivo in arquivos_leituras:
    gravadas_arquivo = gravadas_por_arquivo.get(arquivo, 0)
    registrar_reconciliacao("leituras", arquivo,
                            linhas_arquivo=contar_linhas_arquivo(arquivo),
                            cabecalhos=1,
                            rejeitadas=lidas_por_arquivo[arquivo] - gravadas_arquivo,
                            gravadas=gravadas_arquivo)

## 5. Atendimentos JSON

O array é separado objeto a objeto pelo scanner independente. O Spark lê o
preparo em JSONL com a estrutura aninhada preservada; a Bronze não explode
listas nem aplica regra de atendimento.

In [ ]:
bruto = read_json_com_corrupt("atendimentos.json")
total_lida = bruto.count()
validos, rejeitados = separar_validos_rejeitados(bruto, "atendimentos", "atendimentos.json")
gravadas = write_bronze(validos, "atendimentos", "atendimentos.json")
rejeitadas = total_lida - gravadas
rejeitados_coletados.append(rejeitados)
registrar_reconciliacao("atendimentos", "atendimentos.json",
                        linhas_arquivo=contar_linhas_arquivo("atendimentos.json"),
                        cabecalhos=0, rejeitadas=rejeitadas, gravadas=gravadas,
                        objetos_interpretados=total_lida)

## 6. `inspecoes.xlsx` — 3 abas preservadas
Spark não lê Excel nativamente, então usamos openpyxl e convertemos os campos da planilha para Spark com
**schema explícito string** (evita inferir tipos nos campos do XLSX e que uma coluna vazia
vire `NullType`, que o Delta recusaria gravar). O JSON aninhado mantém a estrutura e os tipos lidos;
`_registro_bruto` preserva o texto de origem. Guardamos `_aba_origem` para não perder de qual
aba cada linha veio.

As 3 abas esperadas são `ago-set`, `out-nov` e `dez`. A Bronze empilha as linhas apenas
para armazenamento e mantém `_aba_origem` em cada registro; isso não afirma que as abas tenham
o mesmo grão nem autoriza compará-las como uma única entidade. A finalidade e o grão de cada aba
continuam pendentes de confirmação com a origem, e o perfil registra essa limitação. Se o conjunto
de abas diferir, a ingestão falha para que a mudança seja analisada antes de alterar a Bronze.

`rejeitadas = 0` aqui é **por construção**: o leitor de Excel (openpyxl) não tem
`badRecordsPath`/`columnNameOfCorruptRecord` — linha ruim dá exceção visível, não é descartada
em silêncio. A reconciliação devolve **3 linhas** (uma por aba, com `_aba_origem`), usa
`linhas_dados = nº de registros lidos na aba` (o conceito de "linha física do arquivo" não se
aplica a um XLSX) e deixa `linhas_arquivo` em `None`.

In [ ]:
import json
from openpyxl import load_workbook

try:
    workbook = load_workbook(f"{RAW}/inspecoes.xlsx", read_only=True, data_only=False)
    abas_encontradas = set(workbook.sheetnames)
    ABAS_ESPERADAS = {"ago-set", "dez", "out-nov"}
    if abas_encontradas != ABAS_ESPERADAS:
        raise ValueError(f"Abas encontradas {sorted(abas_encontradas)}; "
                         f"esperadas {sorted(ABAS_ESPERADAS)}")
    nomes_abas = list(workbook.sheetnames)
    partes = []
    tamanho_por_aba = {}
    for nome_aba in nomes_abas:
        # Empilhamento técnico com aba identificada; não presume grão comum entre abas.
        sheet = workbook[nome_aba]
        linhas = sheet.iter_rows(values_only=True)
        cabecalho = next(linhas, None)
        if not cabecalho:
            raise ValueError(f"Aba sem cabeçalho: {nome_aba}")
        colunas = [str(c) if c is not None else "" for c in cabecalho]
        if (any(not c for c in colunas) or len(set(colunas)) != len(colunas)
                or set(colunas) & AUDIT_FIELDS):
            raise ValueError(f"Cabeçalho vazio, duplicado ou reservado na aba {nome_aba}")
        schema = StructType([StructField(c, StringType(), True) for c in colunas] + [
            StructField("_aba_origem", StringType(), False),
            StructField("_ordem_na_origem", LongType(), False),
            StructField("_linha_fisica_inicial", LongType(), False),
            StructField("_linha_fisica_final", LongType(), False),
            StructField("_registro_bruto", StringType(), True),
        ])
        registros = []
        for linha_fisica, valores in enumerate(linhas, start=2):
            if all(valor is None for valor in valores):
                continue
            if any(valor is not None for valor in valores[len(colunas):]):
                raise ValueError(f"Aba {nome_aba}, linha {linha_fisica}: valor sem cabeçalho")
            celulas = list(valores[:len(colunas)])
            if len(celulas) < len(colunas):
                celulas.extend([None] * (len(colunas) - len(celulas)))
            bruto = json.dumps([{"tipo": type(v).__name__, "valor": v}
                                for v in celulas], ensure_ascii=False, default=str)
            registros.append(tuple(str(v) if v is not None else None for v in celulas) +
                             (nome_aba, len(registros) + 1, linha_fisica, linha_fisica, bruto))
        tamanho_por_aba[nome_aba] = len(registros)
        partes.append(spark.createDataFrame(registros, schema))
        print(f"Aba {nome_aba}: {len(registros):,} registros, {len(colunas)} colunas")
    workbook.close()
except Exception as exc:
    registrar_falha("leitura Excel", "inspecoes.xlsx", exc)
    raise

inspecoes = partes[0]
for parte in partes[1:]:
    inspecoes = inspecoes.unionByName(parte, allowMissingColumns=True)
gravadas = write_bronze(inspecoes, "inspecoes", "inspecoes.xlsx")
gravadas_por_aba = {r["_aba_origem"]: r["count"] for r in
                    spark.table(f"{ALVO_STG}.inspecoes")
                         .groupBy("_aba_origem").count().collect()}
for nome_aba in nomes_abas:
    registrar_reconciliacao("inspecoes", "inspecoes.xlsx",
                            linhas_arquivo=None, cabecalhos=0, rejeitadas=0,
                            gravadas=gravadas_por_aba.get(nome_aba, 0),
                            linhas_dados=tamanho_por_aba[nome_aba], aba_origem=nome_aba)

## 6.1. Rejeitados e reconciliação

Os rejeitados guardam texto bruto, posição, origem e motivo. A reconciliação
usa registros lógicos; linhas físicas aparecem apenas como localização e
diagnóstico, pois um CSV pode ter campos multilinha.

In [ ]:
if rejeitados_coletados:
    rejeitados_df = rejeitados_coletados[0]
    for parte in rejeitados_coletados[1:]:
        rejeitados_df = rejeitados_df.unionByName(parte)

    (rejeitados_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{ALVO_STG}.rejeitados"))
    n_rejeitados_total = spark.table(f"{ALVO_STG}.rejeitados").count()   # conta a tabela gravada, não o DataFrame
    print(f"OK  {ALVO_STG}.rejeitados  {n_rejeitados_total:>10,} registros rejeitados (preparo)")
    rejeitadas_esperadas = sum(r["rejeitadas"] for r in reconciliacao if not r["_tabela"].startswith("_"))
    if n_rejeitados_total != rejeitadas_esperadas:
        raise RuntimeError(f"Quarentena Bronze divergente: {n_rejeitados_total} gravadas em "
                           f"{ALVO_STG}.rejeitados, {rejeitadas_esperadas} esperadas na reconciliação")
else:
    print("Sem fontes com rejeitados a consolidar.")

In [ ]:
recon_schema = StructType([
    StructField("_lote_id", StringType(), False),
    StructField("_lote_origem", StringType(), False),
    StructField("_versao_origem", StringType(), False),
    StructField("_tabela", StringType(), False),
    StructField("_arquivo_origem", StringType(), False),
    StructField("_aba_origem", StringType(), True),
    StructField("hash_sha256", StringType(), True),
    StructField("linhas_arquivo", LongType(), True),
    StructField("cabecalhos", LongType(), False),
    StructField("linhas_dados", LongType(), True),
    StructField("objetos_interpretados", LongType(), True),
    StructField("lidos", LongType(), False),
    StructField("rejeitadas", LongType(), False),
    StructField("gravadas", LongType(), False),
    StructField("duplicatas_detectadas", LongType(), False),
    StructField("corrigidos", LongType(), False),
    StructField("quarentenados", LongType(), False),
    StructField("descartados", LongType(), False),
    StructField("delta", LongType(), True),
    StructField("status", StringType(), False),
    StructField("_data_carga", TimestampType(), False),
])
recon_df = spark.createDataFrame(
    [tuple(r[field.name] for field in recon_schema.fields) for r in reconciliacao],
    recon_schema)
(recon_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{ALVO_STG}.reconciliacao"))
display(recon_df)

In [ ]:
pendentes = [r for r in reconciliacao
             if not r["_tabela"].startswith("_") and r["status"] not in ("OK", "OK_COM_REJEICOES")]
total_rejeitadas = sum(r["rejeitadas"] for r in reconciliacao if not r["_tabela"].startswith("_"))

if pendentes:
    print(f"ATENÇÃO: {len(pendentes)} origem(ns) com reconciliação fora do esperado:")
    for r in pendentes:
        origem = r["_arquivo_origem"] if r["_aba_origem"] is None else f"{r['_arquivo_origem']}[{r['_aba_origem']}]"
        print(f"  - {r['_tabela']} ({origem}): delta={r['delta']} ({r['status']})")
    registrar_falha("reconciliação", "lote", RuntimeError("delta divergente"))
    raise RuntimeError(f"Reconciliação Bronze divergente em {len(pendentes)} origem(ns); "
                       f"consulte {ALVO_STG}.reconciliacao antes de prosseguir. "
                       f"A Bronze publicada em {ALVO} não foi alterada.")
if total_rejeitadas:
    print(f"ATENÇÃO: {total_rejeitadas:,} registro(s) malformado(s) capturado(s) "
          f"→ ver {ALVO_STG}.rejeitados (bruto, com arquivo e lote).")
if not pendentes and not total_rejeitadas :
    print("Reconciliação OK: nenhum registro rejeitado e todas as tabelas fecharam (lidas = gravadas).")
elif not pendentes :
    print("Deltas fechados em todas as tabelas; registros rejeitados foram capturados e preservados "
          f"em {ALVO_STG}.rejeitados.")

## 7. Publicação consistente

Primeiro todas as tabelas Delta históricas recebem o lote e suas contagens
são conferidas. As vistas Bronze leem apenas o ID em `controle_publicacao`;
uma atualização desse ponteiro publica o conjunto de tabelas. Falhas antes
disso deixam o lote anterior visível. O controle registra a execução e o
histórico de reconciliação permanece consultável.

In [ ]:
TABELAS_PUBLICADAS = [
    "unidades_consumidoras", "medidores", "ocorrencias_rede", "geracao_distribuida",
    "faturas", "leituras", "atendimentos", "inspecoes", "rejeitados", "reconciliacao",
]
ALVO_HIST = f"{CATALOG}.bronze_hist"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ALVO_HIST}")

ARQUIVOS_LOTE_SCHEMA = StructType([
    StructField("_lote_id", StringType(), False),
    StructField("_lote_origem", StringType(), False),
    StructField("_versao_origem", StringType(), False),
    StructField("caminho", StringType(), False),
    StructField("nome", StringType(), False),
    StructField("tamanho_bytes", LongType(), False),
    StructField("modificado_em", TimestampType(), False),
    StructField("hash_sha256", StringType(), False),
    StructField("status_hash", StringType(), False),
    StructField("_data_carga", TimestampType(), False),
])


def marcar_lote(status, tabelas, linhas, observacao=None):
    obs = "NULL" if observacao is None else "'" + observacao.replace("'", "''") + "'"
    publicado_em = "current_timestamp()" if status == "PUBLICADO" else "NULL"
    spark.sql(f"""
        MERGE INTO {ALVO}.controle_lote AS d
        USING (SELECT '{LOTE_ID}' AS _lote_id) AS s
        ON d._lote_id = s._lote_id
        WHEN MATCHED THEN UPDATE SET
          publicado_em = {publicado_em}, status = '{status}',
          tabelas_publicadas = {tabelas}, linhas_publicadas = {linhas},
          observacao = {obs}
    """)


def historico_da_tabela(tabela, lote_anterior):
    """Migra a publicação legada uma vez e prepara uma tabela Delta por origem."""
    nome = f"{ALVO_HIST}.{tabela}"
    if spark.catalog.tableExists(nome):
        return
    anterior = f"{ALVO}.{tabela}"
    if spark.catalog.tableExists(anterior):
        dados = spark.table(anterior)
        if "_lote_id" not in dados.columns:
            if not lote_anterior:
                raise RuntimeError(f"{anterior} sem lote identificável; migração bloqueada")
            dados = dados.withColumn("_lote_id", F.lit(lote_anterior))
        else:
            ids = [r[0] for r in dados.select("_lote_id").distinct().limit(2).collect()]
            if ids and ids != [lote_anterior]:
                raise RuntimeError(f"{anterior} não corresponde ao último lote publicado; "
                                   "conferir publicação legada antes de migrar")
    else:
        dados = spark.table(f"{ALVO_STG}.{tabela}").limit(0)
    (dados.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
          .saveAsTable(nome))
    if spark.table(nome).count() != dados.count():
        raise RuntimeError(f"Cópia histórica divergente de {anterior}; migração bloqueada")


def criar_visao_publicada(tabela):
    """Cada consulta filtra a mesma linha ponteiro; lotes incompletos ficam invisíveis."""
    nome = f"{ALVO}.{tabela}"
    # A migração inicial troca a tabela legada após sua cópia ao histórico.
    if spark.catalog.tableExists(nome):
        # SHOW TABLES não expõe o tipo em todos os runtimes; DESCRIBE EXTENDED é estável.
        detalhe = spark.sql(f"DESCRIBE TABLE EXTENDED {nome}").collect()
        kind = next((str(r[1]).strip().upper() for r in detalhe
                     if str(r[0]).strip().lower() == "type"), "")
        if kind != "VIEW":
            spark.sql(f"DROP TABLE {nome}")
    spark.sql(f"""
        CREATE OR REPLACE VIEW {nome} AS
        SELECT h.* FROM {ALVO_HIST}.{tabela} h
        JOIN {ALVO}.controle_publicacao c
          ON c.chave = 'bronze' AND h._lote_id = c.lote_ativo
    """)


try:
    # A tabela de controle é a única operação de troca visível ao consumidor.
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {ALVO}.controle_publicacao (
          chave STRING NOT NULL, lote_ativo STRING, atualizado_em TIMESTAMP
        ) USING DELTA
    """)
    if spark.table(f"{ALVO}.controle_publicacao").count() == 0:
        anterior = spark.sql(f"""
            SELECT _lote_id FROM {ALVO}.controle_lote
            WHERE status = 'PUBLICADO' ORDER BY publicado_em DESC LIMIT 1
        """).collect()
        lote_anterior = anterior[0][0] if anterior else None
        literal = "NULL" if lote_anterior is None else "'" + lote_anterior.replace("'", "''") + "'"
        spark.sql(f"""INSERT INTO {ALVO}.controle_publicacao
                       VALUES ('bronze', {literal}, current_timestamp())""")
    else:
        ativos = spark.sql(f"""SELECT lote_ativo FROM {ALVO}.controle_publicacao
                            WHERE chave = 'bronze'""").collect()
        if len(ativos) != 1:
            raise RuntimeError("Controle de publicação sem uma única linha bronze")
        lote_anterior = ativos[0][0]

    # Primeiro preenche o histórico; falha aqui mantém o ponteiro anterior.
    for tabela in TABELAS_PUBLICADAS:
        historico_da_tabela(tabela, lote_anterior)

    arquivos_df = spark.createDataFrame(
        [tuple(a[field.name] for field in ARQUIVOS_LOTE_SCHEMA.fields)
         for a in arquivos_lote], ARQUIVOS_LOTE_SCHEMA)
    if spark.catalog.tableExists(f"{ALVO}.arquivos_lote"):
        spark.sql(f"DELETE FROM {ALVO}.arquivos_lote WHERE _lote_id = '{LOTE_ID}'")
    (arquivos_df.write.format("delta").mode("append").option("mergeSchema", "true")
        .saveAsTable(f"{ALVO}.arquivos_lote"))

    for tabela in TABELAS_PUBLICADAS:
        nome = f"{ALVO_HIST}.{tabela}"
        spark.sql(f"DELETE FROM {nome} WHERE _lote_id = '{LOTE_ID}'")
        (spark.table(f"{ALVO_STG}.{tabela}").write.format("delta").mode("append")
              .option("mergeSchema", "true").saveAsTable(nome))
        origem = spark.table(f"{ALVO_STG}.{tabela}").count()
        historico = spark.table(nome).where(F.col("_lote_id") == LOTE_ID).count()
        if origem != historico:
            raise RuntimeError(f"Histórico divergente em {tabela}: {origem} != {historico}")

    # Ao migrar uma instalação anterior, cada vista ainda lê o lote antigo.
    for tabela in TABELAS_PUBLICADAS:
        criar_visao_publicada(tabela)

    linhas_publicadas = sum(spark.table(f"{ALVO_STG}.{t}").count()
                           for t in TABELAS_PUBLICADAS)
    marcar_lote("PUBLICADO", len(TABELAS_PUBLICADAS), linhas_publicadas)
    spark.sql(f"""
        MERGE INTO {ALVO}.controle_publicacao d
        USING (SELECT 'bronze' AS chave) s ON d.chave = s.chave
        WHEN MATCHED THEN UPDATE SET lote_ativo = '{LOTE_ID}',
                                     atualizado_em = current_timestamp()
    """)
    print(f"Lote {LOTE_ID} publicado: {len(TABELAS_PUBLICADAS)} fontes, {linhas_publicadas:,} linhas")
except Exception as exc:
    # Uma falha de rede pode ocorrer após o commit; consulte o ponteiro antes
    # de declarar que o lote não foi publicado.
    try:
        atual = spark.sql(f"""SELECT lote_ativo FROM {ALVO}.controle_publicacao
                           WHERE chave = 'bronze'""").first()
    except Exception:
        atual = None
    if atual is None or atual[0] != LOTE_ID:
        marcar_lote("FALHOU", 0, 0, f"{type(exc).__name__}: {exc}"[:500])
    raise

## 8. Conferência final
Lista as tabelas criadas e as contagens. Use isso no print do README.

O schema de destino pode ser compartilhado com outros trabalhos; a contagem abaixo considera
**apenas** as tabelas desta ingestão (as demais não entram nem quebram a conferência).
`bronze_stg` é o preparo do lote: fica no ar como evidência da execução e é sobrescrito na
próxima.

In [ ]:
display(spark.sql(f"SHOW TABLES IN {ALVO}"))
display(spark.sql(f"SHOW TABLES IN {ALVO_STG}"))

# O histórico publicado já está em bronze_hist; o preparo e o JSONL só são necessários até aqui.
# Em falha, esta célula não é alcançada e os artefatos permanecem para diagnóstico.
try:
    spark.sql(f"DROP SCHEMA IF EXISTS {ALVO_STG} CASCADE")
    print(f"Schema temporário removido: {ALVO_STG}")
except Exception as exc:
    print(f"ATENÇÃO: não foi possível remover {ALVO_STG}: {type(exc).__name__}")
try:
    dbutils.fs.rm(f"{RAW}/_bronze_preparo/{LOTE_ID}", True)
    print(f"JSONL temporário removido: {RAW}/_bronze_preparo/{LOTE_ID}")
except Exception as exc:
    print(f"ATENÇÃO: não foi possível remover JSONL temporário: {type(exc).__name__}")


In [ ]:
TABELAS_BRONZE = [
    "unidades_consumidoras", "medidores", "ocorrencias_rede", "geracao_distribuida",
    "faturas", "leituras", "atendimentos", "inspecoes", "reconciliacao", "rejeitados",
    "arquivos_lote", "controle_lote",
]

for nome in TABELAS_BRONZE:
    try:
        print(f"{nome:<26} {spark.table(f'{ALVO}.{nome}').count():>12,} linhas")
    except Exception as e:
        print(f"{nome:<26} AUSENTE/erro: {e}")

## Limites e próxima etapa

Executar no runtime Databricks do Job com os arquivos autorizados e registrar
os resultados no PRD antes de concluir DE-01. O perfil descritivo está no
notebook `Bronze_perfilamento`; correções, imputações e deduplicação ficam na
Silver. O primeiro uso das vistas sobre as tabelas legadas precisa de janela
controlada de migração.